In [0]:
from pyspark.sql import functions as F

In [0]:
SOURCE_PATH = "abfss://raw@fintechdllasya.dfs.core.windows.net/transactions/2026-08-13/"

In [0]:
SCHEMA_LOCATION = "abfss://raw@fintechdllasya.dfs.core.windows.net/_autoloader_schema/transactions/"

df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .load(SOURCE_PATH)
)

In [0]:
df.printSchema()

In [0]:
CHECKPOINT_PATH = "abfss://raw@fintechdllasya.dfs.core.windows.net/_autoloader_checkpoint/transactions/"

In [0]:
BRONZE_TABLE = "dbx_fintech_data_platform.bronze.transactions_autoloader"

In [0]:
query = (
    df.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(availableNow= True)
    .toTable(BRONZE_TABLE)
)

In [0]:
query = (
    df.writeStream
      .format("delta")
      .option("checkpointLocation", CHECKPOINT_PATH)
      .outputMode("append")
      .trigger(availableNow=True)
      .toTable(BRONZE_TABLE)
)

In [0]:
NEW_TXN_SOURCE = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "transactions/2026-08-13/transactions_00.csv"
)

new_txn_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(NEW_TXN_SOURCE)
)

new_txn_df.count()

In [0]:
new_txn_df = new_txn_df.withColumn(
    "transaction_id",
    F.regexp_replace(
        F.col("transaction_id"),
        "^T20260813_",
        "T20260814_"
    )
)
print("Total rows:", new_txn_df.count())
print("Unique IDs:", new_txn_df.select("transaction_id").distinct().count())

In [0]:
NEW_FILE_PATH = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "transactions/2026-08-13/new_transactions_24.csv"
)

(
    new_txn_df
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(NEW_FILE_PATH)
)

In [0]:
display(dbutils.fs.ls(
    "abfss://raw@fintechdllasya.dfs.core.windows.net/transactions/2026-08-13/"
))

In [0]:
display(dbutils.fs.ls(NEW_FILE_PATH))

In [0]:
parts = [
    f.path
    for f in dbutils.fs.ls(NEW_FILE_PATH)
    if f.name.endswith(".csv")
]

parts

In [0]:
single_file_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(parts)
)

print("Rows:", single_file_df.count())
print("Unique IDs:", single_file_df.select("transaction_id").distinct().count())

In [0]:
TEMP_OUTPUT = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "transactions/temp_new_transaction/"
)

(
    single_file_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(TEMP_OUTPUT)
)

In [0]:
display(dbutils.fs.ls(TEMP_OUTPUT))

In [0]:
display(dbutils.fs.ls(SOURCE_PATH))

In [0]:
NEW_FILE = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "transactions/2026-08-13/new_transactions_24.csv"
)

part_file = [
    f.path
    for f in dbutils.fs.ls(TEMP_OUTPUT)
    if f.name.endswith(".csv")
][0]

dbutils.fs.cp(part_file, NEW_FILE)

In [0]:
display(dbutils.fs.ls(SOURCE_PATH))

In [0]:
query = (
    df.writeStream
      .format("delta")
      .option("checkpointLocation", CHECKPOINT_PATH)
      .outputMode("append")
      .trigger(availableNow=True)
      .toTable(BRONZE_TABLE)
)

In [0]:
spark.sql(f"""
SELECT COUNT(*) AS total_records
FROM {BRONZE_TABLE}
""").show()

In [0]:
df_with_metadata = (
    df
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn(
        "_source_date",
        F.to_date(
            F.regexp_extract(
                F.col("_metadata.file_path"),
                r"/transactions/(\d{4}-\d{2}-\d{2})/",
                1
            )
        )
    )
)

In [0]:
df_with_metadata.printSchema()

In [0]:
BRONZE_TABLE_V2 = "dbx_fintech_data_platform.bronze.transactions_autoloader_v2"

CHECKPOINT_PATH_V2 = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "_autoloader_checkpoint/transactions_v2/"
)

In [0]:
query_v2 = (
    df_with_metadata.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT_PATH_V2)
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE_V2)
)

In [0]:
spark.sql(f"""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT transaction_id) AS unique_transactions,
    COUNT(DISTINCT _source_file) AS source_files,
    COUNT(DISTINCT _source_date) AS source_dates
FROM {BRONZE_TABLE_V2}
""").show()

In [0]:
display(
    spark.sql(f"""
        SELECT
            transaction_id,
            _source_file,
            _source_date,
            _ingestion_timestamp
        FROM {BRONZE_TABLE_V2}
        ORDER BY _ingestion_timestamp DESC
        LIMIT 10
    """)
)

In [0]:
spark.sql(f"""
SELECT
    COUNT(*) AS rows_from_new_file,
    COUNT(DISTINCT transaction_id) AS unique_transactions
FROM {BRONZE_TABLE_V2}
WHERE _source_file LIKE '%new_transactions_24.csv'
""").show()

In [0]:
SCHEMA_TEST_PATH = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "transactions/schema_test/"
)

In [0]:
schema_change_df = (
    new_txn_df
    .withColumn("risk_score", F.lit(0.85))
    .limit(10)
)

In [0]:
schema_change_df.printSchema()

In [0]:
(
    schema_change_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(SCHEMA_TEST_PATH)
)

display(dbutils.fs.ls(SCHEMA_TEST_PATH))

In [0]:
SCHEMA_TEST_LOCATION = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "_autoloader_schema/schema_test/"
)

SCHEMA_TEST_CHECKPOINT = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "_autoloader_checkpoint/schema_test/"
)

schema_test_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_TEST_LOCATION)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(SCHEMA_TEST_PATH)
)

In [0]:
schema_test_df.printSchema()

In [0]:
SCHEMA_EVOLUTION_PATH = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "transactions/schema_evolution_test/"
)

SCHEMA_EVOLUTION_LOCATION = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "_autoloader_schema/schema_evolution_test/"
)

SCHEMA_EVOLUTION_CHECKPOINT = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "_autoloader_checkpoint/schema_evolution_test/"
)

SCHEMA_EVOLUTION_TABLE = (
    "dbx_fintech_data_platform.bronze.transactions_schema_test"
)

In [0]:
baseline_df = new_txn_df.limit(10)

In [0]:
(
    baseline_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(SCHEMA_EVOLUTION_PATH)
)

In [0]:
baseline_df = new_txn_df.limit(10)

In [0]:
schema_test_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_EVOLUTION_LOCATION)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(SCHEMA_EVOLUTION_PATH)
)

In [0]:
schema_test_query = (
    schema_test_stream.writeStream
        .format("delta")
        .option("checkpointLocation", SCHEMA_EVOLUTION_CHECKPOINT)
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(SCHEMA_EVOLUTION_TABLE)
)

In [0]:
spark.sql(f"""
SELECT COUNT(*) AS records
FROM {SCHEMA_EVOLUTION_TABLE}
""").show()

In [0]:
SCHEMA_CHANGE_TEMP = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "transactions/schema_evolution_temp/"
)

schema_change_df = (
    new_txn_df
    .withColumn("risk_score", F.lit(0.85))
    .limit(10)
)

(
    schema_change_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(SCHEMA_CHANGE_TEMP)
)

In [0]:
schema_change_part = [
    f.path
    for f in dbutils.fs.ls(SCHEMA_CHANGE_TEMP)
    if f.name.endswith(".csv")
][0]

schema_change_part

In [0]:
SCHEMA_CHANGE_FILE = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "transactions/schema_evolution_test/schema_change.csv"
)

dbutils.fs.cp(schema_change_part, SCHEMA_CHANGE_FILE)

In [0]:
display(dbutils.fs.ls(SCHEMA_EVOLUTION_PATH))

In [0]:
schema_test_query = (
    schema_test_stream.writeStream
        .format("delta")
        .option("checkpointLocation", SCHEMA_EVOLUTION_CHECKPOINT)
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(SCHEMA_EVOLUTION_TABLE)
)

In [0]:
schema_test_stream.printSchema()

In [0]:
schema_test_stream_v2 = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_EVOLUTION_LOCATION)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(SCHEMA_EVOLUTION_PATH)
)

In [0]:
schema_test_query_v2 = (
    schema_test_stream_v2.writeStream
        .format("delta")
        .option("checkpointLocation", SCHEMA_EVOLUTION_CHECKPOINT)
        .option("mergeSchema", "true")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(SCHEMA_EVOLUTION_TABLE)
)

In [0]:
spark.sql(f"DESCRIBE TABLE {SCHEMA_EVOLUTION_TABLE}").show(truncate=False)

In [0]:
spark.sql(f"""
SELECT
    COUNT(*) AS total_records,
    COUNT(risk_score) AS records_with_risk_score
FROM {SCHEMA_EVOLUTION_TABLE}
""").show()

In [0]:
TEST_TRANSACTION_FILE = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "transactions/2026-08-13/new_transactions_24.csv"
)

dbutils.fs.rm(TEST_TRANSACTION_FILE, True)

In [0]:
display(dbutils.fs.ls(SOURCE_PATH))

In [0]:
SOURCE_PATH = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "transactions/"
)

SCHEMA_LOCATION = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "_autoloader_schema/transactions_final/"
)

CHECKPOINT_LOCATION = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "_autoloader_checkpoint/transactions_final/"
)

BRONZE_TABLE = (
    "dbx_fintech_data_platform.bronze.transactions_autoloader"
)

In [0]:
TEST_PATHS = [
    "abfss://raw@fintechdllasya.dfs.core.windows.net/transactions/schema_test/",
    "abfss://raw@fintechdllasya.dfs.core.windows.net/transactions/schema_evolution_test/",
    "abfss://raw@fintechdllasya.dfs.core.windows.net/transactions/schema_evolution_temp/",
    "abfss://raw@fintechdllasya.dfs.core.windows.net/transactions/temp_new_transaction/"
]

for path in TEST_PATHS:
    dbutils.fs.rm(path, True)

In [0]:
display(
    dbutils.fs.ls(
        "abfss://raw@fintechdllasya.dfs.core.windows.net/transactions/2026-08-13/"
    )
)

In [0]:
import pyspark.sql.functions as F

transactions_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(SOURCE_PATH)
)

In [0]:
transactions_stream.printSchema()

In [0]:
transactions_stream_with_metadata = (
    transactions_stream
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn(
        "_source_date",
        F.to_date(
            F.regexp_extract(
                F.col("_metadata.file_path"),
                r"/transactions/(\d{4}-\d{2}-\d{2})/",
                1
            )
        )
    )
)

In [0]:
transactions_query = (
    transactions_stream_with_metadata.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT_LOCATION)
        .option("mergeSchema", "true")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

In [0]:
spark.sql(f"""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT transaction_id) AS unique_transactions,
    COUNT(DISTINCT _source_file) AS source_files,
    COUNT(DISTINCT _source_date) AS source_dates
FROM {BRONZE_TABLE}
""").show()

In [0]:
spark.sql(f"""
SELECT
    _source_file,
    COUNT(*) AS row_count,
    COUNT(DISTINCT transaction_id) AS unique_transactions
FROM {BRONZE_TABLE}
GROUP BY _source_file
ORDER BY row_count DESC
""").show(30, truncate=False)

In [0]:
spark.sql(f"""
SELECT
    version,
    timestamp,
    operation,
    operationMetrics['numOutputRows'] AS output_rows,
    operationMetrics['numAddedFiles'] AS added_files
FROM (DESCRIBE HISTORY {BRONZE_TABLE})
WHERE operation = 'STREAMING UPDATE'
ORDER BY version
""").show(truncate=False)

In [0]:
spark.sql(f"""
DROP TABLE IF EXISTS {BRONZE_TABLE}
""")

In [0]:
dbutils.fs.rm(
    CHECKPOINT_LOCATION,
    True
)

In [0]:
transactions_query = (
    transactions_stream_with_metadata.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT_LOCATION)
        .option("mergeSchema", "true")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

In [0]:
spark.sql(f"""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT transaction_id) AS unique_transactions,
    COUNT(DISTINCT _source_file) AS source_files,
    COUNT(DISTINCT _source_date) AS source_dates
FROM {BRONZE_TABLE}
""").show()

In [0]:
spark.sql(f"""
SELECT
    _source_file,
    COUNT(*) AS row_count,
    COUNT(DISTINCT transaction_id) AS unique_transactions
FROM {BRONZE_TABLE}
GROUP BY _source_file
ORDER BY _source_file
""").show(30, truncate=False)

In [0]:
from pyspark.sql import functions as F

audit_df = spark.table(
    "dbx_fintech_data_platform.metadata.ingestion_log"
)

audit_df.printSchema()

In [0]:
display(
    audit_df.orderBy(
        F.col("started_at").desc()
    )
)

### Auto Loader + Audit Framework

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid

In [0]:
SOURCE_PATH = "abfss://raw@fintechdllasya.dfs.core.windows.net/transactions/"

SCHEMA_LOCATION = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "_autoloader_schema/transactions_final/"
)

CHECKPOINT_LOCATION = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "_autoloader_checkpoint/transactions_final/"
)

BRONZE_TABLE = "dbx_fintech_data_platform.bronze.transactions_autoloader_final"

RUN_AUDIT_TABLE = (
    "dbx_fintech_data_platform.metadata.pipeline_run_log"
)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {RUN_AUDIT_TABLE} (
    run_id STRING,
    pipeline_name STRING,
    target_table STRING,
    status STRING,
    rows_processed BIGINT,
    started_at TIMESTAMP,
    completed_at TIMESTAMP,
    error_message STRING
)
USING DELTA
""")

In [0]:
spark.sql(f"""
DESCRIBE TABLE {RUN_AUDIT_TABLE}
""").show()

In [0]:
RUN_ID = str(uuid.uuid4())

PIPELINE_NAME = "transaction_autoloader_ingestion"

STARTED_AT = datetime.now(timezone.utc)

print(f"Run ID       : {RUN_ID}")
print(f"Pipeline     : {PIPELINE_NAME}")
print(f"Started At   : {STARTED_AT}")

In [0]:
start_audit_df = spark.createDataFrame(
    [
        (
            RUN_ID,
            PIPELINE_NAME,
            BRONZE_TABLE,
            "RUNNING",
            0,
            STARTED_AT,
            None,
            None
        )
    ],
    """
    run_id STRING,
    pipeline_name STRING,
    target_table STRING,
    status STRING,
    rows_processed BIGINT,
    started_at TIMESTAMP,
    completed_at TIMESTAMP,
    error_message STRING
    """
)

start_audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(RUN_AUDIT_TABLE)

In [0]:
spark.sql(f"""
SELECT *
FROM {RUN_AUDIT_TABLE}
WHERE run_id = '{RUN_ID}'
""").show(truncate=False)

In [0]:
transactions_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(SOURCE_PATH)
)

In [0]:
transactions_stream_with_metadata = (
    transactions_stream
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_source_date",
        F.to_date(
            F.regexp_extract(
                F.col("_metadata.file_path"),
                r"/transactions/(\d{4}-\d{2}-\d{2})/",
                1
            )
        )
    )
)

In [0]:
def process_transaction_batch(batch_df, batch_id):

    batch_start = datetime.now(timezone.utc)

    try:

        # Count records in the current micro-batch
        batch_count = batch_df.count()

        print(
            f"Run ID: {RUN_ID} | "
            f"Batch ID: {batch_id} | "
            f"Rows: {batch_count}"
        )

        # Write batch to Bronze
        (
            batch_df.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(BRONZE_TABLE)
        )

        print(
            f"Batch {batch_id} successfully written "
            f"to {BRONZE_TABLE}"
        )

    except Exception as e:

        print(
            f"Batch {batch_id} failed: {str(e)}"
        )

        raise

In [0]:
def process_transaction_batch(batch_df, batch_id):

    batch_start = datetime.now(timezone.utc)

    try:

        batch_count = batch_df.count()

        print(
            f"Run ID: {RUN_ID} | "
            f"Batch ID: {batch_id} | "
            f"Rows: {batch_count}"
        )

        # Write data to Bronze
        (
            batch_df.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(BRONZE_TABLE)
        )

        # Record successful batch
        batch_audit_df = spark.createDataFrame(
            [
                (
                    RUN_ID,
                    PIPELINE_NAME,
                    BRONZE_TABLE,
                    "SUCCESS",
                    batch_count,
                    batch_start,
                    datetime.now(timezone.utc),
                    None
                )
            ],
            """
            run_id STRING,
            pipeline_name STRING,
            target_table STRING,
            status STRING,
            rows_processed BIGINT,
            started_at TIMESTAMP,
            completed_at TIMESTAMP,
            error_message STRING
            """
        )

        batch_audit_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(RUN_AUDIT_TABLE)

    except Exception as e:

        error_message = str(e)

        failed_audit_df = spark.createDataFrame(
            [
                (
                    RUN_ID,
                    PIPELINE_NAME,
                    BRONZE_TABLE,
                    "FAILED",
                    0,
                    batch_start,
                    datetime.now(timezone.utc),
                    error_message
                )
            ],
            """
            run_id STRING,
            pipeline_name STRING,
            target_table STRING,
            status STRING,
            rows_processed BIGINT,
            started_at TIMESTAMP,
            completed_at TIMESTAMP,
            error_message STRING
            """
        )

        failed_audit_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(RUN_AUDIT_TABLE)

        raise

In [0]:
# spark.sql(f"""
# ALTER TABLE {RUN_AUDIT_TABLE}
# ADD COLUMNS (
#     batch_id BIGINT
# )
# """)

In [0]:
def process_transaction_batch(batch_df, batch_id):

    batch_start = datetime.now(timezone.utc)

    try:

        batch_count = batch_df.count()

        print(
            f"Run ID: {RUN_ID} | "
            f"Batch ID: {batch_id} | "
            f"Rows: {batch_count}"
        )

        # Write data to Bronze
        (
            batch_df.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(BRONZE_TABLE)
        )

        # Write successful audit record
        batch_audit_df = spark.createDataFrame(
            [
                (
                    RUN_ID,
                    PIPELINE_NAME,
                    BRONZE_TABLE,
                    "SUCCESS",
                    batch_count,
                    batch_start,
                    datetime.now(timezone.utc),
                    None,
                    int(batch_id)
                )
            ],
            """
            run_id STRING,
            pipeline_name STRING,
            target_table STRING,
            status STRING,
            rows_processed BIGINT,
            started_at TIMESTAMP,
            completed_at TIMESTAMP,
            error_message STRING,
            batch_id BIGINT
            """
        )

        batch_audit_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(RUN_AUDIT_TABLE)

    except Exception as e:

        error_message = str(e)

        failed_audit_df = spark.createDataFrame(
            [
                (
                    RUN_ID,
                    PIPELINE_NAME,
                    BRONZE_TABLE,
                    "FAILED",
                    0,
                    batch_start,
                    datetime.now(timezone.utc),
                    error_message,
                    int(batch_id)
                )
            ],
            """
            run_id STRING,
            pipeline_name STRING,
            target_table STRING,
            status STRING,
            rows_processed BIGINT,
            started_at TIMESTAMP,
            completed_at TIMESTAMP,
            error_message STRING,
            batch_id BIGINT
            """
        )

        failed_audit_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(RUN_AUDIT_TABLE)

        raise

In [0]:
transactions_query = (
    transactions_stream_with_metadata.writeStream
        .foreachBatch(process_transaction_batch)
        .option(
            "checkpointLocation",
            CHECKPOINT_LOCATION
        )
        .outputMode("append")
        .trigger(availableNow=True)
        .start()
)

transactions_query.awaitTermination()

In [0]:
COMPLETED_AT = datetime.now(timezone.utc)

spark.sql(f"""
UPDATE {RUN_AUDIT_TABLE}
SET
    status = 'SUCCESS',
    rows_processed = (
        SELECT COALESCE(SUM(rows_processed), 0)
        FROM {RUN_AUDIT_TABLE}
        WHERE run_id = '{RUN_ID}'
          AND status = 'SUCCESS'
    ),
    completed_at = TIMESTAMP('{COMPLETED_AT.strftime("%Y-%m-%d %H:%M:%S")}')
WHERE run_id = '{RUN_ID}'
  AND status = 'RUNNING'
""")

In [0]:
RUN_ID = str(uuid.uuid4())
PIPELINE_NAME = "transaction_autoloader_ingestion"
STARTED_AT = datetime.now(timezone.utc)

# Insert RUNNING record
start_audit_df = spark.createDataFrame(
    [
        (
            RUN_ID,
            PIPELINE_NAME,
            BRONZE_TABLE,
            "RUNNING",
            0,
            STARTED_AT,
            None,
            None,
            None
        )
    ],
    """
    run_id STRING,
    pipeline_name STRING,
    target_table STRING,
    status STRING,
    rows_processed BIGINT,
    started_at TIMESTAMP,
    completed_at TIMESTAMP,
    error_message STRING,
    batch_id BIGINT
    """
)

start_audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(RUN_AUDIT_TABLE)

try:

    transactions_query = (
        transactions_stream_with_metadata.writeStream
            .foreachBatch(process_transaction_batch)
            .option(
                "checkpointLocation",
                CHECKPOINT_LOCATION
            )
            .outputMode("append")
            .trigger(availableNow=True)
            .start()
    )

    transactions_query.awaitTermination()

    COMPLETED_AT = datetime.now(timezone.utc)

    spark.sql(f"""
    UPDATE {RUN_AUDIT_TABLE}
    SET
        status = 'SUCCESS',
        rows_processed = (
            SELECT COALESCE(SUM(rows_processed), 0)
            FROM {RUN_AUDIT_TABLE}
            WHERE run_id = '{RUN_ID}'
              AND status = 'SUCCESS'
        ),
        completed_at = TIMESTAMP(
            '{COMPLETED_AT.strftime("%Y-%m-%d %H:%M:%S")}'
        )
    WHERE run_id = '{RUN_ID}'
      AND status = 'RUNNING'
    """)

except Exception as e:

    ERROR_MESSAGE = str(e)
    COMPLETED_AT = datetime.now(timezone.utc)

    spark.sql(f"""
    UPDATE {RUN_AUDIT_TABLE}
    SET
        status = 'FAILED',
        completed_at = TIMESTAMP(
            '{COMPLETED_AT.strftime("%Y-%m-%d %H:%M:%S")}'
        ),
        error_message = '{ERROR_MESSAGE.replace("'", "''")}'
    WHERE run_id = '{RUN_ID}'
      AND status = 'RUNNING'
    """)

    raise

In [0]:
spark.sql(f"""
SELECT
    run_id,
    pipeline_name,
    target_table,
    status,
    rows_processed,
    started_at,
    completed_at,
    error_message,
    batch_id
FROM {RUN_AUDIT_TABLE}
WHERE run_id = '{RUN_ID}'
ORDER BY batch_id
""").show(truncate=False)

In [0]:
spark.sql(f"""
SELECT
    SUM(rows_processed) AS total_audited_rows
FROM {RUN_AUDIT_TABLE}
WHERE run_id = '{RUN_ID}'
  AND status = 'SUCCESS'
""").show()

In [0]:
spark.sql("""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT transaction_id) AS unique_transactions,
    COUNT(DISTINCT _source_file) AS source_files
FROM dbx_fintech_data_platform.bronze.transactions_autoloader_v2
""").show()

In [0]:
# spark.sql(f"""
# SELECT
#     COUNT(*) AS total_records,
#     COUNT(DISTINCT transaction_id) AS unique_transactions,
#     COUNT(DISTINCT _source_file) AS source_files,
#     COUNT(DISTINCT _source_date) AS source_dates
# FROM {BRONZE_TABLE}
# """).show()

In [0]:
spark.sql("""
CREATE TABLE dbx_fintech_data_platform.bronze.transactions_autoloader_final
AS
SELECT *
FROM dbx_fintech_data_platform.bronze.transactions_autoloader
""")

In [0]:
spark.sql("""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT transaction_id) AS unique_transactions,
    COUNT(DISTINCT _source_file) AS source_files,
    COUNT(DISTINCT _source_date) AS source_dates
FROM dbx_fintech_data_platform.bronze.transactions_autoloader_final
""").show()

In [0]:
spark.sql("""
SELECT *
FROM dbx_fintech_data_platform.metadata.pipeline_run_log
ORDER BY started_at DESC
""").show(truncate=False)